# Hey Score Demo — Par B
## Datathon Hey Banco 2026

## 0. Setup & Imports

In [ ]:
# Instalar dependencias
!pip install gradio google-genai --quiet

# Imports
import os
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
import seaborn as sns
from google.colab import userdata, drive
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print("Imports listos")

## 1. Conexión a Google Drive

In [ ]:
drive.mount('/content/drive')

# Rutas base
BASE_PATH = '/content/drive/MyDrive/hey_datathon'
DATA_PATH = f'{BASE_PATH}/data'
OUTPUTS_PATH = f'{BASE_PATH}/outputs'
FIGURES_PATH = f'{BASE_PATH}/figures'

print(f" Drive montado")
print(f" Data: {DATA_PATH}")
print(f" Outputs: {OUTPUTS_PATH}")
print(f" Figures: {FIGURES_PATH}")

## 2. Carga de Datos

In [ ]:
# Carga de datos
clientes = pd.read_csv(f'{DATA_PATH}/hey_clientes.csv')
productos = pd.read_csv(f'{DATA_PATH}/hey_productos.csv')

print(f"clientes: {clientes.shape}")
print(f"productos: {productos.shape}")

In [ ]:
# Cargar archivo master_df
master_df = pd.read_parquet(f'{OUTPUTS_PATH}/master_df.parquet')
score_per_user = pd.read_csv(f'{OUTPUTS_PATH}/score_per_user.csv')

print(f"master_df: {master_df.shape}")
print(f"columnas: {list(master_df.columns)}")

### 2.1 Verificar valores en usuarios

In [ ]:
# Checar si las dimensiones tiene valores
dims = ['dim_solidez', 'dim_diversificacion', 'dim_gasto', 'dim_engagement', 'dim_proteccion']
for d in dims:
    validos = master_df[d].notna().sum()
    print(f"{d}: {validos} validos de 15025")

## 3. Mock de Usuario

In [ ]:
def get_usuario(user_id):
    row = master_df[master_df['user_id'] == user_id].iloc[0]
    return {
        "user_id": row['user_id'],
        "hey_score": round(row['hey_score'], 1),
        "dim_solidez": round(row['dim_solidez'], 1),
        "dim_diversificacion": round(row['dim_diversificacion'], 1),
        "dim_gasto": round(row['dim_gasto'], 1),
        "dim_engagement": round(row['dim_engagement'], 1),
        "dim_proteccion": round(row['dim_proteccion'], 1),
        "segmento": row['cluster_etiqueta'],
        "tema_dominante": row['tema_dominante'],
        "sentimiento_promedio": round(row['sentimiento_promedio'], 2),
        "edad": int(row['edad']),
        "ocupacion": row['ocupacion'],
        "ingreso_mensual_mxn": int(row['ingreso_mensual_mxn']),
        "es_hey_pro": bool(row['es_hey_pro']),
        "score_buro": int(row['score_buro']),
    }

# Prueba con un usuario real
usuario = get_usuario('USR-00042')
print(usuario)

## 4. Demo Gradio

### 4.1 Dimensiones de cards

In [ ]:
def card_dimensiones(usuario):
    dims = ['Solidez', 'Diversificación', 'Gasto', 'Engagement', 'Protección']
    keys = ['dim_solidez', 'dim_diversificacion', 'dim_gasto',
            'dim_engagement', 'dim_proteccion']
    maximo = 20

    fig, ax = plt.subplots(figsize=(11.5, 3))
    fig.patch.set_facecolor('#12121e')
    ax.set_facecolor('#12121e')
    ax.axis('off')

    for i, (dim, key) in enumerate(zip(dims, keys)):
        val   = usuario.get(key, 0) or 0
        pct   = val / maximo
        y     = 1 - (i * 0.19) - 0.08
        color = '#FF6B35' if pct >= 0.7 else '#f59e0b' if pct >= 0.4 else '#ef4444'

        ax.plot(0.025, y, 'o', color=color, markersize=9, zorder=5,
                transform=ax.transAxes, clip_on=False)

        ax.text(0.05, y, dim,
                transform=ax.transAxes, va='center', ha='left',
                color='#ccc', fontsize=11, fontweight='500')

        ax.barh(y, 0.63, height=0.10,
                left=0.22, align='center', color='#1e1e30',
                transform=ax.transAxes)
        ax.barh(y, pct * 0.63, height=0.10,
                left=0.22, align='center', color=color, alpha=0.9,
                transform=ax.transAxes)

        ax.text(0.98, y, f'{val:.1f}/20',
                transform=ax.transAxes, va='center', ha='right',
                color=color, fontsize=11, fontweight='700',
                fontfamily='monospace')

    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    buf = io.BytesIO()
    plt.savefig(buf, format='png',
            facecolor=fig.get_facecolor(), dpi=100)
    buf.seek(0)
    img = Image.open(buf)
    plt.close()
    return img

### 4.2 Image Placeholder

In [ ]:
def imagen_placeholder(titulo, subtitulo, ancho=5.5, alto=5.5):
    fig, ax = plt.subplots(figsize=(ancho, alto))
    fig.patch.set_facecolor('#12121e')
    ax.set_facecolor('#12121e')
    ax.axis('off')

    # Borde decorativo discreto
    ax.text(0.5, 0.62, "○",
            transform=ax.transAxes, ha='center', va='center',
            fontsize=70, color='#1e1e30', fontweight='100')

    # Título
    ax.text(0.5, 0.50, titulo,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=14, color='#555', fontweight='700',
            fontfamily='monospace')

    # Subtítulo con instrucción
    ax.text(0.5, 0.40, subtitulo,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=10, color='#333', fontstyle='italic')

    # Línea decorativa naranja sutil abajo
    ax.axhline(0.25, color='#FF6B35', linewidth=1, alpha=0.3,
               xmin=0.40, xmax=0.60)

    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    buf = io.BytesIO()
    plt.savefig(buf, format='png',
                facecolor=fig.get_facecolor(), dpi=100)
    buf.seek(0)
    img = Image.open(buf)
    plt.close()
    return img

# Pre-generar las 3 imágenes — se calculan una sola vez
PLACEHOLDER_PERFIL = imagen_placeholder("PERFIL", "Ingresa un User ID")
PLACEHOLDER_RADAR  = imagen_placeholder("RADAR", "5 dimensiones del Hey Score")
PLACEHOLDER_BARS   = imagen_placeholder("DIMENSIONES", "Análisis detallado", ancho=11.5, alto=3)

### 4.3 Mostrar Perfil

In [ ]:
def mostrar_perfil(user_id):
    uid = user_id.strip()
    if uid in CACHE_IMAGENES:
        c = CACHE_IMAGENES[uid]
        return c['perfil'], c['radar'], c['bars'], c['havi'], c['cf']
    # Fallback para IDs arbitrarios
    usuario = get_usuario(uid)
    return (
        card_perfil(usuario),
        radar_chart(usuario),
        card_dimensiones(usuario),
        generar_havi_texto(usuario),
        counterfactual_hey_pro(uid),
    )

#### Precargar

In [ ]:
CACHE_IMAGENES = {}

def precargar_usuario(user_id):
    usuario = get_usuario(user_id)
    CACHE_IMAGENES[user_id] = {
        'perfil': card_perfil(usuario),
        'radar': radar_chart(usuario),
        'bars': card_dimensiones(usuario),
        'havi': generar_havi_texto(usuario),
        'cf': counterfactual_hey_pro(user_id),
    }

# Pre-generar los 3 perfiles de demo
for uid in ['USR-14024', 'USR-12651', 'USR-03003']:
    precargar_usuario(uid)

### 4.4 Card Perfil

In [ ]:
def card_perfil(usuario):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    fig.patch.set_facecolor('#12121e')
    ax.set_facecolor('#12121e')
    ax.axis('off')

    score = usuario['hey_score']
    color = '#FF6B35' if score >= 70 else '#f59e0b' if score >= 45 else '#ef4444'

    # Score grande
    ax.text(0.5, 0.92, f"{score:.0f}",
            transform=ax.transAxes, ha='center', va='top',
            fontsize=56, fontweight='700', color=color,
            fontfamily='monospace')
    ax.text(0.5, 0.72, "/ 100",
            transform=ax.transAxes, ha='center', va='top',
            fontsize=13, color='#555', fontfamily='monospace')
    segmento_limpio = ''.join(c for c in usuario['segmento'] if c.isascii() or c.isalpha()).strip()
    ax.text(0.5, 0.66, segmento_limpio,
            transform=ax.transAxes, ha='center', va='top',
            fontsize=12, color='white', fontweight='600')

    # Separador
    ax.axhline(0.56, color='#1e1e30', linewidth=1, xmin=0.08, xmax=0.92)

    # Dos columnas de datos
    left_labels = ['ID', 'EDAD', 'OCUPACIÓN', 'INGRESO']
    left_values = [
        usuario['user_id'],
        f"{usuario['edad']} años",
        usuario['ocupacion'],
        f"${usuario['ingreso_mensual_mxn']:,} MXN",
    ]
    right_labels = ['TEMA', 'SENTIMIENTO', 'HEY PRO', 'SCORE BURÓ']
    right_values = [
        usuario['tema_dominante'],
        f"{usuario['sentimiento_promedio']:+.2f}",
        'Sí' if usuario['es_hey_pro'] else 'No',
        str(usuario.get('score_buro', '—')),
    ]

    for i, (lbl, val) in enumerate(zip(left_labels, left_values)):
        y = 0.48 - i * 0.12
        ax.text(0.06, y, lbl,
                transform=ax.transAxes, ha='left', va='top',
                fontsize=8, color='#444', fontweight='700',
                fontfamily='monospace')
        ax.text(0.06, y - 0.05, val,
                transform=ax.transAxes, ha='left', va='top',
                fontsize=10, color='#ddd', fontweight='500')

    for i, (lbl, val) in enumerate(zip(right_labels, right_values)):
        y = 0.48 - i * 0.12
        ax.text(0.55, y, lbl,
                transform=ax.transAxes, ha='left', va='top',
                fontsize=8, color='#444', fontweight='700',
                fontfamily='monospace')
        val_color = '#ddd'
        if lbl == 'SENTIMIENTO':
            val_color = '#ef4444' if usuario['sentimiento_promedio'] < -0.2 else '#4CAF50'
        ax.text(0.55, y - 0.05, val,
                transform=ax.transAxes, ha='left', va='top',
                fontsize=10, color=val_color, fontweight='500')

    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    buf = io.BytesIO()
    plt.savefig(buf, format='png',
            facecolor=fig.get_facecolor(), dpi=200)
    buf.seek(0)
    img = Image.open(buf)
    plt.close()
    return img

### 4.5 Graficas

In [ ]:
def radar_chart(usuario):
    dims = ['Solidez', 'Diversificación', 'Gasto', 'Engagement', 'Protección']
    valores = [
        usuario['dim_solidez'],
        usuario['dim_diversificacion'],
        usuario['dim_gasto'],
        usuario['dim_engagement'],
        usuario['dim_proteccion'],
    ]
    valores = [v if not np.isnan(v) else 0 for v in valores]

    N            = len(dims)
    angulos      = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    valores_plot = valores + [valores[0]]
    angulos_plot = angulos + [angulos[0]]

    fig, ax = plt.subplots(figsize=(5.5, 5.5), subplot_kw=dict(polar=True))
    fig.patch.set_facecolor('#12121e')
    ax.set_facecolor('#12121e')

    ax.set_ylim(0, 20)
    ax.set_yticks([5, 10, 15, 20])
    ax.set_yticklabels(['5', '10', '15', '20'], color='#333', fontsize=7)
    ax.yaxis.grid(True, color='#1e1e30', linewidth=1)
    ax.xaxis.grid(True, color='#1e1e30', linewidth=1)
    ax.spines['polar'].set_color('#1e1e30')

    score = usuario['hey_score']
    color = '#FF6B35' if score >= 70 else '#f59e0b' if score >= 45 else '#ef4444'

    ax.fill(angulos_plot, valores_plot, color=color, alpha=0.15)
    ax.plot(angulos_plot, valores_plot, color=color, linewidth=2.2)
    ax.scatter(angulos, valores, color=color, s=40, zorder=5)

    ax.set_xticks(angulos)
    ax.set_xticklabels(dims, color='#ccc', fontsize=10, fontweight='600')
    ax.tick_params(pad=10)

    plt.subplots_adjust(left=0.15, right=0.85, top=0.88, bottom=0.12)
    buf = io.BytesIO()
    plt.savefig(buf, format='png',
            facecolor=fig.get_facecolor(), dpi=100)
    buf.seek(0)
    img = Image.open(buf)
    plt.close()
    return img

In [ ]:
def grafica_distribucion():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0f0f13')

    colores_cluster = {
        '💎 Cliente consolidado': '#FF6B35',
        '🚨 Usuario en riesgo':   '#ef4444',
        '⚠️ Usuario medio':       '#f59e0b',
        '😴 Usuario inactivo':    '#6366f1',
        '🌱 Recién bancarizado':  '#4CAF50',
    }

    # ── Gráfica 1: Histograma del Hey Score ──
    ax1 = axes[0]
    ax1.set_facecolor('#1a1a2e')
    ax1.hist(master_df['hey_score'], bins=30, color='#FF6B35',
             alpha=0.85, edgecolor='#0f0f13', linewidth=0.5)
    media = master_df['hey_score'].mean()
    ax1.axvline(media, color='white', linestyle='--', linewidth=1.5, alpha=0.7)
    ax1.text(media + 1, ax1.get_ylim()[1] * 0.9,
             f'Media: {media:.1f}', color='white', fontsize=9)
    ax1.set_title('Distribución del Hey Score', color='white',
                  fontsize=13, fontweight='bold', pad=12)
    ax1.set_xlabel('Hey Score', color='#aaa', fontsize=10)
    ax1.set_ylabel('Usuarios', color='#aaa', fontsize=10)
    ax1.tick_params(colors='#aaa')
    for spine in ax1.spines.values():
        spine.set_edgecolor('#2a2a3a')

    # ── Gráfica 2: Usuarios por cluster ──
    ax2 = axes[1]
    ax2.set_facecolor('#1a1a2e')
    conteo = master_df['cluster_etiqueta'].value_counts()
    colores = [colores_cluster.get(c, '#888') for c in conteo.index]
    bars = ax2.barh(conteo.index, conteo.values,
                    color=colores, alpha=0.85, edgecolor='#0f0f13', linewidth=0.5)

    # Etiquetas dentro de las barras
    for bar, val in zip(bars, conteo.values):
        ax2.text(val * 0.5, bar.get_y() + bar.get_height() / 2,
                 f'{val:,}', va='center', ha='center',
                 color='white', fontsize=9, fontweight='bold')

    ax2.set_title('Usuarios por Segmento', color='white',
                  fontsize=13, fontweight='bold', pad=12)
    ax2.set_xlabel('Número de usuarios', color='#aaa', fontsize=10)
    ax2.tick_params(colors='#aaa')
    for spine in ax2.spines.values():
        spine.set_edgecolor('#2a2a3a')

    plt.tight_layout(pad=2)
    buf = io.BytesIO()
    plt.savefig(buf,
            format='png',
            facecolor=fig.get_facecolor(),
            dpi=150)
    buf.seek(0)
    img = Image.open(buf)
    plt.close()
    return img

### 4.6 Havi

In [ ]:
def generar_havi_texto(usuario: dict) -> str:
    cluster = usuario.get('segmento', '')
    score = usuario.get('hey_score', 0)
    sentimiento  = usuario.get('sentimiento_promedio', 0)
    tema = usuario.get('tema_dominante', '')
    ocupacion = usuario.get('ocupacion', '')
    ingreso = usuario.get('ingreso_mensual_mxn', 0)
    dim_solidez = usuario.get('dim_solidez', 0)
    dim_proteccion = usuario.get('dim_proteccion', 0)

    if '💎' in cluster:
        if dim_proteccion < 40:
            return (
                f"Tu Hey Score es {score:.0f} — estás entre nuestros clientes más sólidos. "
                f"El único punto a reforzar es tu protección: un seguro de compras o de vida "
                f"puede completar tu perfil financiero."
            )
        return (
            f"Tu Hey Score es {score:.0f} — excelente posición financiera. "
            f"Tu solidez ({dim_solidez:.0f}/100) está por encima del promedio. "
            f"Considera mover parte de tu saldo a una inversión Hey para generar rendimiento."
        )

    elif '🚨' in cluster:
        return (
            f"Hola, tu Hey Score es {score:.0f} y detectamos presión en tu crédito. "
            f"Un pago parcial esta semana puede reducir tu utilización y mejorar tu score "
            f"de forma inmediata. Estamos aquí para ayudarte."
        )

    elif '⚠️' in cluster:
        if sentimiento < -0.1:
            return (
                f"Tu Hey Score es {score:.0f}. Notamos que has tenido algunas fricciones "
                f"recientes relacionadas con {tema}. Queremos resolverlas — "
                f"¿en qué podemos ayudarte hoy?"
            )
        return (
            f"Tu Hey Score es {score:.0f}. Usas activamente tu crédito, "
            f"que es una buena señal. El siguiente paso es diversificar: "
            f"un producto de inversión o seguro puede subir tu score varios puntos."
        )

    elif '😴' in cluster:
        return (
            f"Tu Hey Score es {score:.0f}. Tienes cuenta con nosotros pero poca actividad reciente. "
            f"¿Sabías que domiciliar tu nómina aquí como {ocupacion.lower()} "
            f"activa beneficios desde el primer depósito?"
        )

    elif '🌱' in cluster:
        return (
            f"Tu Hey Score es {score:.0f} — estás construyendo tu historial financiero. "
            f"Con movimientos constantes en los próximos 3 meses tu score puede subir "
            f"más de 10 puntos. Cada transacción cuenta."
        )

    else:
        return f"Tu Hey Score es {score:.0f}. Estoy aquí para ayudarte a mejorar tu salud financiera con Hey Banco."

#### Generar Havi

In [ ]:
def generar_havi(user_id):
    try:
        usuario = get_usuario(user_id.strip())
    except:
        return "Usuario no encontrado"
    return generar_havi_texto(usuario)

### 4.7 Counterfactual

In [ ]:
DELTAS_HEY_PRO = {
    'dim_solidez': 1.54,
    'dim_diversificacion': 2.38,
    'dim_gasto': 0.25,
    'dim_engagement': 4.97,
    'dim_proteccion': 3.02,
    'hey_score': 12.16
}

def counterfactual_hey_pro(user_id):
    try:
        usuario = get_usuario(user_id.strip())
    except:
        return "Usuario no encontrado"

    if usuario.get('es_hey_pro'):
        return f"✅ {usuario['user_id']} ya tiene Hey Pro activo — su score de {usuario['hey_score']:.0f} ya incluye estos beneficios."

    score_actual = usuario['hey_score']
    score_proyectado = min(score_actual + DELTAS_HEY_PRO['hey_score'], 100)

    return (
        f"📊 Counterfactual — ¿Qué pasaría si {usuario['user_id']} activara Hey Pro?\n\n"
        f"Hey Score actual:     {score_actual:.0f}\n"
        f"Hey Score proyectado: {score_proyectado:.0f}  (+{score_proyectado - score_actual:.0f} puntos)\n\n"
        f"Cambios por dimensión:\n"
        f"  Engagement:       +{DELTAS_HEY_PRO['dim_engagement']:.1f} pts\n"
        f"  Diversificación:  +{DELTAS_HEY_PRO['dim_diversificacion']:.1f} pts\n"
        f"  Protección:       +{DELTAS_HEY_PRO['dim_proteccion']:.1f} pts\n"
        f"  Solidez:          +{DELTAS_HEY_PRO['dim_solidez']:.1f} pts\n"
        f"  Gasto:            +{DELTAS_HEY_PRO['dim_gasto']:.1f} pts"
    )

### 4.8 Gradio

#### CSS

In [ ]:
css = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Space+Mono:wght@400;700&display=swap');

:root, html, body, .gradio-container,
.dark, gradio-app {
    background: #080810 !important;
    color: #f0f0f0 !important;
}

html.dark, html:not(.dark) {
    background: #080810 !important;
}

/* Forzar tema oscuro en todos los componentes */
.app, .main, .contain, .wrap, .panel {
    background: #080810 !important;
}

*, body, .gradio-container {
    font-family: 'DM Sans', sans-serif !important;
}

body {
    background: #080810 !important;
}

.gradio-container {
    background: #080810 !important;
    max-width: 1280px !important;
    margin: auto !important;
    padding: 0 24px 40px !important;
}

/* ── Tabs ── */
.tab-nav {
    background: transparent !important;
    border: none !important;
    border-bottom: 1px solid #1e1e30 !important;
    padding: 0 !important;
    margin-bottom: 24px !important;
    gap: 4px !important;
}
.tab-nav button {
    color: #555 !important;
    font-weight: 600 !important;
    font-size: 0.9em !important;
    padding: 10px 20px !important;
    border-radius: 0 !important;
    border: none !important;
    border-bottom: 2px solid transparent !important;
    background: transparent !important;
    transition: all 0.2s !important;
    letter-spacing: 0.3px !important;
}
.tab-nav button:hover {
    color: #FF6B35 !important;
}
.tab-nav button.selected {
    color: #FF6B35 !important;
    border-bottom: 2px solid #FF6B35 !important;
    background: transparent !important;
}

/* ── Inputs ── */
input[type="text"], textarea {
    background: #12121e !important;
    border: 1px solid #1e1e30 !important;
    color: #f0f0f0 !important;
    border-radius: 10px !important;
    font-family: 'DM Sans', sans-serif !important;
    font-size: 0.95em !important;
    transition: border-color 0.2s !important;
}
input[type="text"]:focus, textarea:focus {
    border-color: #FF6B35 !important;
    box-shadow: 0 0 0 3px rgba(255,107,53,0.12) !important;
    outline: none !important;
}

/* ── Labels ── */
label > span, .block > label > span {
    color: #666 !important;
    font-size: 0.75em !important;
    font-weight: 700 !important;
    text-transform: uppercase !important;
    letter-spacing: 1px !important;
}

/* ── Blocks / Cards ── */
.block {
    background: #12121e !important;
    border: 1px solid #1e1e30 !important;
    border-radius: 16px !important;
    padding: 20px !important;
    transition: border-color 0.2s !important;
}
.block:hover {
    border-color: #2a2a3e !important;
}

/* ── Botones ── */
#btn-buscar {
    background: #FF6B35 !important;
    border: none !important;
    color: white !important;
    font-weight: 700 !important;
    border-radius: 10px !important;
    font-size: 0.95em !important;
    letter-spacing: 0.3px !important;
    box-shadow: 0 0 24px rgba(255,107,53,0.25) !important;
    transition: all 0.2s !important;
}
#btn-buscar:hover {
    background: #ff8254 !important;
    box-shadow: 0 0 32px rgba(255,107,53,0.4) !important;
    transform: translateY(-1px) !important;
}
#btn-havi {
    background: transparent !important;
    border: 1px solid #FF6B35 !important;
    color: #FF6B35 !important;
    font-weight: 600 !important;
    border-radius: 10px !important;
    transition: all 0.2s !important;
}
#btn-havi:hover {
    background: rgba(255,107,53,0.08) !important;
}
#btn-cf {
    background: transparent !important;
    border: 1px solid #4CAF50 !important;
    color: #4CAF50 !important;
    font-weight: 600 !important;
    border-radius: 10px !important;
    transition: all 0.2s !important;
}
#btn-cf:hover {
    background: rgba(76,175,80,0.08) !important;
}

/* ── Dataframe — versión reforzada ── */
.dataframe,
table.dataframe,
.gradio-dataframe,
.gradio-dataframe table,
[data-testid="dataframe"] table {
    border-radius: 12px !important;
    overflow: hidden !important;
    border: 1px solid #1e1e30 !important;
    background: #12121e !important;
    color: #ddd !important;
    border-collapse: separate !important;
    border-spacing: 0 !important;
}

.dataframe thead th,
table.dataframe thead th,
.gradio-dataframe thead th,
[data-testid="dataframe"] thead th,
.svelte-1m4rwwn thead th {
    background: #1e1e30 !important;
    color: #FF6B35 !important;
    font-weight: 700 !important;
    text-transform: uppercase !important;
    font-size: 0.75em !important;
    letter-spacing: 0.8px !important;
    padding: 12px 16px !important;
    border: none !important;
    border-bottom: 1px solid #2a2a3e !important;
}

.dataframe tbody td,
table.dataframe tbody td,
.gradio-dataframe tbody td,
[data-testid="dataframe"] tbody td,
.svelte-1m4rwwn tbody td {
    background: #12121e !important;
    color: #ddd !important;
    border-color: #1e1e30 !important;
    border-top: 1px solid #1e1e30 !important;
    padding: 10px 16px !important;
    font-size: 0.9em !important;
}

.dataframe tbody tr:hover td,
table.dataframe tbody tr:hover td,
.gradio-dataframe tbody tr:hover td,
[data-testid="dataframe"] tbody tr:hover td {
    background: #1a1a2e !important;
    color: white !important;
}

/* Contenedor del dataframe */
.gradio-dataframe,
[data-testid="dataframe"],
div:has(> table.dataframe) {
    background: #12121e !important;
    border-radius: 12px !important;
}

/* Scrollbar interno del dataframe */
.dataframe::-webkit-scrollbar-track,
.gradio-dataframe::-webkit-scrollbar-track {
    background: #12121e !important;
}

/* ── Imágenes ── */
img {
    border-radius: 14px !important;
    border: 1px solid #1e1e30 !important;
}

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 6px; height: 6px; }
::-webkit-scrollbar-track { background: #12121e; }
::-webkit-scrollbar-thumb { background: #2a2a3e; border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: #FF6B35; }
"""

#### Gradio

In [ ]:
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

# Card del caso estrella — calculada una vez
caso_estrella = get_usuario('USR-13714')

# Stats para header
total_usuarios = len(master_df)
score_promedio = master_df['hey_score'].mean()

def cargar_demo1():
  uid = "USR-14024"
  perfil, radar, bars, havi, cf = mostrar_perfil(uid)
  return uid, perfil, radar, bars, havi, cf

def cargar_demo2():
  uid = "USR-12651"
  perfil, radar, bars, havi, cf = mostrar_perfil(uid)
  return uid, perfil, radar, bars, havi, cf

def cargar_demo3():
  uid = "USR-03003"
  perfil, radar, bars, havi, cf = mostrar_perfil(uid)
  return uid, perfil, radar, bars, havi, cf

with gr.Blocks(title="Hey Score — Hey Banco 2026", css=css,
               theme=gr.themes.Base()) as demo:

    # ── Header ──────────────────────────────────────────────────────────
    gr.HTML(f"""
    <div style="padding: 40px 0 32px; text-align: center;">
        <div style="display:inline-block; background:rgba(255,107,53,0.1);
                    border:1px solid rgba(255,107,53,0.3); border-radius:20px;
                    padding:4px 14px; font-size:0.78em; color:#FF6B35;
                    font-weight:700; letter-spacing:1px; text-transform:uppercase;
                    margin-bottom:16px;">
            Datathon Hey Banco 2026
        </div>
        <h1 style="font-family:'Space Mono',monospace; font-size:3.2em;
                   color:#ffffff; margin:0; letter-spacing:-2px; line-height:1;">
            Hey <span style="color:#FF6B35;">Score</span>
        </h1>
        <p style="color:#555; font-size:0.95em; margin:10px 0 28px;">
            Sistema de salud financiera proactiva · Powered by ML + IA
        </p>
        <div style="display:flex; justify-content:center; gap:12px; flex-wrap:wrap;">
            <div style="background:#12121e; border:1px solid #1e1e30;
                        border-radius:12px; padding:14px 24px; min-width:140px;">
                <div style="font-family:'Space Mono',monospace; font-size:1.8em;
                            color:#FF6B35; font-weight:700;">{total_usuarios:,}</div>
                <div style="color:#555; font-size:0.75em; font-weight:700;
                            text-transform:uppercase; letter-spacing:0.8px; margin-top:4px;">
                    Usuarios analizados</div>
            </div>
            <div style="background:#12121e; border:1px solid #1e1e30;
                        border-radius:12px; padding:14px 24px; min-width:140px;">
                <div style="font-family:'Space Mono',monospace; font-size:1.8em;
                            color:#4CAF50; font-weight:700;">{score_promedio:.1f}</div>
                <div style="color:#555; font-size:0.75em; font-weight:700;
                            text-transform:uppercase; letter-spacing:0.8px; margin-top:4px;">
                    Score promedio</div>
            </div>
            <div style="background:#12121e; border:1px solid #1e1e30;
                        border-radius:12px; padding:14px 24px; min-width:140px;">
                <div style="font-family:'Space Mono',monospace; font-size:1.8em;
                            color:#6366f1; font-weight:700;">5</div>
                <div style="color:#555; font-size:0.75em; font-weight:700;
                            text-transform:uppercase; letter-spacing:0.8px; margin-top:4px;">
                    Segmentos detectados</div>
            </div>
        </div>
    </div>
    """)

    with gr.Tabs():

        # ── TAB 1 ────────────────────────────────────────────────────────
        with gr.Tab("🔍  Análisis individual"):
            gr.Markdown("**Perfiles de demo**")
            with gr.Row():
              btn_demo1 = gr.Button("💎 USR-14024 · Cliente consolidado")
              btn_demo2 = gr.Button("🚨 USR-12651 · Usuario en riesgo")
              btn_demo3 = gr.Button("🌱 USR-03003 · Recién bancarizado")

            with gr.Row():
                user_input = gr.Textbox(
                    label="User ID", placeholder="ej. USR-13714", scale=4)
                btn_perfil = gr.Button(
                    "Analizar →", variant="primary", scale=1, elem_id="btn-buscar")

            with gr.Row():
                with gr.Column():
                    perfil_output = gr.Image(label="Perfil financiero", height=450, value=PLACEHOLDER_PERFIL, show_download_button=False,
        show_fullscreen_button=False,
)

                with gr.Column():
                    radar_output = gr.Image(label="Radar de dimensiones", height=450, value=PLACEHOLDER_RADAR, show_download_button=False,
        show_fullscreen_button=False,
)

            with gr.Row():
                bars_output = gr.Image(label="Detalle por dimensión", height=300, value=PLACEHOLDER_BARS, show_download_button=False,
        show_fullscreen_button=False,
)

            with gr.Row():
                with gr.Column():
                    havi_output = gr.Textbox(
                        label="Havi — Recomendación proactiva", lines=5, value="")
                    btn_havi = gr.Button("✨  Generar recomendación", elem_id="btn-havi")
                with gr.Column():
                    cf_output = gr.Textbox(
                        label="Simulación — Si activa Hey Pro", lines=5, value="")
                    btn_cf = gr.Button("📈  Simular Hey Pro", elem_id="btn-cf")

            btn_perfil.click(fn=mostrar_perfil, inputs=user_input,
                             outputs=[perfil_output, radar_output, bars_output,
                                      havi_output, cf_output])
            btn_havi.click(fn=generar_havi, inputs=user_input, outputs=havi_output)
            btn_cf.click(fn=counterfactual_hey_pro, inputs=user_input, outputs=cf_output)


            btn_demo1.click(fn=cargar_demo1, outputs=[user_input, perfil_output, radar_output, bars_output, havi_output, cf_output])
            btn_demo2.click(fn=cargar_demo2, outputs=[user_input, perfil_output, radar_output, bars_output, havi_output, cf_output])
            btn_demo3.click(fn=cargar_demo3, outputs=[user_input, perfil_output, radar_output, bars_output, havi_output, cf_output])

        # ── TAB 2 ────────────────────────────────────────────────────────
        with gr.Tab("📊  Distribución del Score"):
            gr.HTML("""
            <div style="margin-bottom:20px;">
                <h3 style="color:white; font-size:1.1em; font-weight:700; margin:0 0 6px;">
                    Vista general del dataset
                </h3>
                <p style="color:#555; font-size:0.88em; margin:0;">
                    Distribución del Hey Score entre los 15,025 usuarios,
                    segmentada por cluster.
                </p>
            </div>
            """)

            gr.Image(value=grafica_distribucion(), label="", show_label=False)

            stats_cluster = master_df.groupby('cluster_etiqueta')['hey_score'].agg(
                ['count', 'mean', 'min', 'max']
            ).round(1).reset_index()
            stats_cluster.columns = ['Segmento', 'Usuarios', 'Score promedio', 'Mínimo', 'Máximo']

            gr.Dataframe(
                value       = stats_cluster,
                label       = "Estadísticas por segmento",
                interactive = False,
            )

demo.launch(share=True, prevent_thread_lock=False)

## 5 Clientes para la demo

In [ ]:
for segmento in ['💎 Cliente consolidado', '🚨 Usuario en riesgo', '🌱 Recién bancarizado']:
    subset = master_df[master_df['cluster_etiqueta'] == segmento]
    ejemplo = subset.sort_values('hey_score', ascending=(segmento != '💎 Cliente consolidado')).iloc[0]
    print(f"{segmento}")
    print(f"  user_id: {ejemplo['user_id']}")
    print(f"  hey_score: {ejemplo['hey_score']:.1f}")
    print(f"  es_hey_pro: {ejemplo['es_hey_pro']}")
    print(f"  utilizacion_promedio: {ejemplo['utilizacion_promedio']:.2f}")
    print()

## 6 Graficas de distribucion

### 1. Distribución del Hey Score

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(master_df['hey_score'], kde=True, color='#FF6B35')
plt.title('Distribución del Hey Score', fontsize=16)
plt.xlabel('Hey Score', fontsize=12)
plt.ylabel('Frecuencia', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### 2. Boxplots por Dimensión

In [ ]:
dims = ['dim_solidez', 'dim_diversificacion', 'dim_gasto', 'dim_engagement', 'dim_proteccion']
plt.figure(figsize=(15, 8))
sns.boxplot(data=master_df[dims], palette='viridis')
plt.title('Boxplots de las Dimensiones del Hey Score', fontsize=16)
plt.ylabel('Valor de la Dimensión', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### 3. Top Categorías Dominantes

In [ ]:
plt.figure(figsize=(12, 7))
sns.countplot(y='categoria_dominante', data=master_df, order=master_df['categoria_dominante'].value_counts().index, palette='plasma')
plt.title('Distribución de Categorías Dominantes', fontsize=16)
plt.xlabel('Número de Usuarios', fontsize=12)
plt.ylabel('Categoría Dominante', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

### 4. Scatter Plot: Ingreso Mensual vs. Hey Score

In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(x='ingreso_mensual_mxn', y='hey_score', data=master_df, hue='cluster_etiqueta', palette='coolwarm', alpha=0.7)
plt.title('Ingreso Mensual vs. Hey Score', fontsize=16)
plt.xlabel('Ingreso Mensual (MXN)', fontsize=12)
plt.ylabel('Hey Score', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xscale('log') # Escala logarítmica para ingresos si hay mucha variación
plt.show()

### 5. Distribución por Cluster de Usuario

In [ ]:
plt.figure(figsize=(12, 7))
sns.countplot(y='cluster_etiqueta', data=master_df, order=master_df['cluster_etiqueta'].value_counts().index, palette='viridis')
plt.title('Distribución de Usuarios por Cluster', fontsize=16)
plt.xlabel('Número de Usuarios', fontsize=12)
plt.ylabel('Etiqueta del Cluster', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

### 6. Distribución de Edad

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(master_df['edad'], kde=True, bins=20, color='skyblue')
plt.title('Distribución de Edad de los Usuarios', fontsize=16)
plt.xlabel('Edad', fontsize=12)
plt.ylabel('Frecuencia', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()